# Introduction to RPC Pseudo-Tracklets

## Ilirjan Margjeka, Universiteti Luarasi
## Klaudio Peqini,   Universiteti i Tiranes

# This notebook is a **student-friendly companion** to the simplified RPC pseudo-tracklet repo.
Its goal is not to reproduce the full research pipeline, but to help students understand the **physical idea**, the **mathematical logic**, and a **minimal computational implementation**.

We will proceed slowly:

1. What is the detector problem?
2. What is a pseudo-tracklet?
3. How do we define **eligible** and **matched** events?
4. How do we estimate the efficiency?
5. How can a simple efficiency depend on the high voltage?
6. What can students modify and explore on their own?

At the end, there is a short set of **exercises**.

## 1. Physical motivation

A Resistive Plate Chamber (RPC) is a gaseous detector that produces signals when a charged particle crosses it.  
When several RPC planes are arranged one after another, the information from some planes can be used to **predict**
where the particle should appear in another plane.

This leads to a very useful idea:

- use some RPCs as **reference chambers**
- use another RPC as the **target chamber**
- compare the **predicted** position with the **measured** position
- count how often the target chamber responds correctly

This gives a simple estimate of detector efficiency.

## 2. The simplified geometric model

We consider three RPC planes:

- **RPC A** at position $z_A$
- **RPC B** at position $z_B$
- **RPC C** at position $z_C$

A particle crosses the system approximately along a straight line.  
If we know the hit positions in A and B, we can estimate the expected hit position in C.

In this notebook we use a strongly simplified model:

- motion is described only in the $x$ direction
- the particle trajectory is assumed to be linear
- timing is represented by a single scalar variable
- detector imperfections are represented by random noise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Detector plane positions (arbitrary units, e.g. cm)
zA, zB, zC = 0.0, 50.0, 100.0

print(f"RPC A at z = {zA}")
print(f"RPC B at z = {zB}")
print(f"RPC C at z = {zC}")

## 3. Visualizing the detector layout

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

ax.axvline(zA, ymin=0.15, ymax=0.85, linewidth=2, label="RPC A")
ax.axvline(zB, ymin=0.15, ymax=0.85, linewidth=2, label="RPC B")
ax.axvline(zC, ymin=0.15, ymax=0.85, linewidth=2, label="RPC C")

ax.text(zA, 1.02, "A", ha="center")
ax.text(zB, 1.02, "B", ha="center")
ax.text(zC, 1.02, "C", ha="center")

ax.set_xlim(-10, 110)
ax.set_ylim(0, 1.1)
ax.set_xlabel("z position")
ax.set_yticks([])
ax.set_title("Simplified three-RPC geometry")
plt.show()

## 4. Simulating particle events

To build an educational model, we generate artificial events.

For each event we assign:

- an initial position $x_0$
- a slope $m$
- a reference time $t_0$

From these we compute the ideal positions at A, B, and C:

$$
x(z) = x_0 + mz.
$$

Then we add measurement noise, which imitates detector uncertainty.

In [ ]:
def simulate_events(
    n_events=5000,
    sigma_x=0.6,
    sigma_t=0.8,
    true_efficiency=0.92,
    seed=42
):
    rng = np.random.default_rng(seed)

    x0 = rng.uniform(-20, 20, n_events)
    m = rng.normal(0, 0.05, n_events)
    t0 = rng.normal(0, 1.0, n_events)

    # Ideal positions
    xA_true = x0 + m * zA
    xB_true = x0 + m * zB
    xC_true = x0 + m * zC

    # Measured positions in reference chambers
    xA_meas = xA_true + rng.normal(0, sigma_x, n_events)
    xB_meas = xB_true + rng.normal(0, sigma_x, n_events)

    # Measured times in reference chambers
    tA_meas = t0 + rng.normal(0, sigma_t, n_events)
    tB_meas = t0 + rng.normal(0, sigma_t, n_events)

    # Whether the target chamber responds
    detected_in_C = rng.random(n_events) < true_efficiency

    # Measured C only when detected
    xC_meas = np.where(
        detected_in_C,
        xC_true + rng.normal(0, sigma_x, n_events),
        np.nan
    )
    tC_meas = np.where(
        detected_in_C,
        t0 + rng.normal(0, sigma_t, n_events),
        np.nan
    )

    return pd.DataFrame({
        "xA": xA_meas, "xB": xB_meas, "xC": xC_meas,
        "tA": tA_meas, "tB": tB_meas, "tC": tC_meas,
        "detected_in_C": detected_in_C
    })

df = simulate_events()
df.head()

## 5. Predicting the target hit from the reference chambers

Using A and B, we estimate the straight line and extrapolate it to C.

If the measured positions in A and B are $x_A$ and $x_B$, then the slope estimate is

$$
m_{\mathrm{est}} = \frac{x_B - x_A}{z_B - z_A}.
$$

The predicted position at C is

$$
x_C^{\mathrm{pred}} = x_A + m_{\mathrm{est}}(z_C - z_A).
$$

For timing, we use a very simple approximation: the expected target time is the average of the reference times,

$$
t_C^{\mathrm{pred}} \approx \frac{t_A + t_B}{2}.
$$

In [ ]:
def predict_target(df):
    m_est = (df["xB"] - df["xA"]) / (zB - zA)
    xC_pred = df["xA"] + m_est * (zC - zA)
    tC_pred = 0.5 * (df["tA"] + df["tB"])
    out = df.copy()
    out["xC_pred"] = xC_pred
    out["tC_pred"] = tC_pred
    return out

dfp = predict_target(df)
dfp[["xA", "xB", "xC", "xC_pred", "tA", "tB", "tC", "tC_pred"]].head()

## 6. Eligible and matched events

This is the most important conceptual step.

### Eligible event
An event is **eligible** if the reference chambers provide enough information to make a prediction.

In this simplified notebook, every event with valid A and B measurements is eligible.

### Matched event
An event is **matched** if the target chamber records a hit compatible with the prediction.

We introduce tolerance windows:

- spatial tolerance: $\Delta x < \mathrm{tol}_x$
- timing tolerance: $\Delta t < \mathrm{tol}_t$

Thus the matching rule is:

$$
|x_C^{\mathrm{meas}} - x_C^{\mathrm{pred}}| < \mathrm{tol}_x
\quad\text{and}\quad
|t_C^{\mathrm{meas}} - t_C^{\mathrm{pred}}| < \mathrm{tol}_t.
$$

In [ ]:
tol_x = 2.0
tol_t = 2.0

def classify_events(df, tol_x=2.0, tol_t=2.0):
    out = df.copy()

    out["eligible"] = out["xA"].notna() & out["xB"].notna()

    out["dx"] = np.abs(out["xC"] - out["xC_pred"])
    out["dt"] = np.abs(out["tC"] - out["tC_pred"])

    out["matched"] = (
        out["eligible"]
        & out["xC"].notna()
        & out["tC"].notna()
        & (out["dx"] < tol_x)
        & (out["dt"] < tol_t)
    )
    return out

dfc = classify_events(dfp, tol_x=tol_x, tol_t=tol_t)
dfc[["eligible", "matched", "dx", "dt"]].head()

## 7. Efficiency estimate

The efficiency is estimated as

$$
\varepsilon = \frac{N_{\mathrm{matched}}}{N_{\mathrm{eligible}}}.
$$

This ratio is the core observable of the notebook.

In [ ]:
N_eligible = int(dfc["eligible"].sum())
N_matched = int(dfc["matched"].sum())
eff = N_matched / N_eligible

print(f"Eligible events : {N_eligible}")
print(f"Matched events  : {N_matched}")
print(f"Estimated efficiency = {eff:.4f}")

## 8. Inspecting the residuals

It is useful to look at the distributions of:

- position residual: $x_C^{\mathrm{meas}} - x_C^{\mathrm{pred}}$
- time residual: $t_C^{\mathrm{meas}} - t_C^{\mathrm{pred}}$

These plots help us understand whether the tolerances are too strict or too loose.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
res_x = (dfc["xC"] - dfc["xC_pred"]).dropna()
ax.hist(res_x, bins=40)
ax.axvline(-tol_x, linestyle="--")
ax.axvline(tol_x, linestyle="--")
ax.set_xlabel("x residual")
ax.set_ylabel("Counts")
ax.set_title("Residual distribution in position")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
res_t = (dfc["tC"] - dfc["tC_pred"]).dropna()
ax.hist(res_t, bins=40)
ax.axvline(-tol_t, linestyle="--")
ax.axvline(tol_t, linestyle="--")
ax.set_xlabel("t residual")
ax.set_ylabel("Counts")
ax.set_title("Residual distribution in time")
plt.show()

## 9. A single-event visual example

Let us draw one event to make the idea concrete.

In [ ]:
example = dfc.dropna(subset=["xC"]).iloc[0]

z_vals = np.array([zA, zB, zC])
x_meas = np.array([example["xA"], example["xB"], example["xC"]])
x_pred = np.array([example["xA"], example["xB"], example["xC_pred"]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z_vals[:2], x_meas[:2], "o-", label="Reference measurements (A,B)")
ax.plot([zB, zC], [example["xB"], example["xC_pred"]], "--", label="Predicted continuation")
ax.plot(zC, example["xC"], "s", markersize=8, label="Measured hit in C")
ax.axhspan(example["xC_pred"] - tol_x, example["xC_pred"] + tol_x, alpha=0.2)

ax.set_xlabel("z")
ax.set_ylabel("x")
ax.set_title("Single-event pseudo-tracklet example")
ax.legend()
plt.show()

print("Matched?" , bool(example["matched"]))

## 10. A simple HV scan

In real detector studies, the efficiency is measured as a function of the applied high voltage (HV).  
Here we simulate that behavior in a simplified way.

We assume that the underlying detection probability in the target chamber increases with HV according to a sigmoid-like law.

In [ ]:
def true_efficiency_from_hv(hv, hv50=6.85, k=10.0, plateau=0.96):
    return plateau / (1 + np.exp(-k * (hv - hv50)))

hvs = np.array([6.0, 6.5, 6.7, 6.8, 6.9, 7.0, 7.1, 7.2, 7.3, 7.4, 7.5])

records = []

for hv in hvs:
    eff_true = true_efficiency_from_hv(hv)
    dfi = simulate_events(n_events=4000, true_efficiency=eff_true, seed=int(hv * 1000))
    dfi = predict_target(dfi)
    dfi = classify_events(dfi, tol_x=tol_x, tol_t=tol_t)

    N_el = int(dfi["eligible"].sum())
    N_ma = int(dfi["matched"].sum())
    eff_est = N_ma / N_el
    err = np.sqrt(eff_est * (1 - eff_est) / N_el)

    records.append({
        "HV_kV": hv,
        "true_efficiency": eff_true,
        "estimated_efficiency": eff_est,
        "binomial_error": err
    })

scan = pd.DataFrame(records)
scan

## 11. Plotting the HV scan

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    scan["HV_kV"],
    scan["estimated_efficiency"],
    yerr=scan["binomial_error"],
    fmt="o",
    capsize=4,
    label="Estimated efficiency"
)
ax.plot(scan["HV_kV"], scan["true_efficiency"], "--", label="Underlying model")

ax.set_xlabel("High Voltage (kV)")
ax.set_ylabel("Efficiency")
ax.set_title("Simplified efficiency vs HV")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 12. Interpretation

The previous plot shows the main idea of an HV scan:

- at low voltage, the chamber is not fully efficient
- as the voltage increases, the efficiency rises
- eventually the curve reaches a plateau

This type of behavior is central in detector operation studies.

## 13. What was simplified here?

This notebook is intentionally pedagogical.  
Compared with a research-grade workflow, we simplified many things:

- we used only one spatial coordinate ($x$)
- we assumed straight trajectories
- we ignored strip-level clustering details
- we used a toy timing model
- we simulated data rather than reading ROOT files
- we treated reconstruction and detector effects in a minimal way

Even so, the essential logic remains:

> reference chambers define a prediction,  
> the target chamber is checked against that prediction,  
> and efficiency is the fraction of successful matches.

## 14. Optional extensions for students

Students can extend the notebook in many ways:

1. Add a second spatial coordinate $y$
2. Introduce a fourth chamber
3. Compare two-chamber and three-chamber reconstruction
4. Change the tolerance windows and study the effect on efficiency
5. Simulate more realistic noise
6. Build a 2D efficiency map in the detector plane
7. Replace the toy simulation with real detector files in a later project

# Exercises for students

## Exercise 1
Change the spatial tolerance `tol_x`.  
Try at least three values, for example:

- `tol_x = 1.0`
- `tol_x = 2.0`
- `tol_x = 4.0`

What happens to the estimated efficiency?  
Explain physically why that happens.

---

## Exercise 2
Change the timing tolerance `tol_t`.  
Does the result behave in the same way as for `tol_x`?  
Why or why not?

---

## Exercise 3
Modify the measurement noise `sigma_x` in `simulate_events`.  
How does worsening the detector resolution affect the residual distributions and the final efficiency estimate?

---

## Exercise 4
Compare two models:

- Model A: use only RPC A to guess the hit in C in a very naive way
- Model B: use RPC A and RPC B together, as done in this notebook

Why is Model B better?

---

## Exercise 5
Change the number of simulated events.  
Compare the uncertainty on the efficiency for:

- 500 events
- 2000 events
- 10000 events

What happens to the binomial error?  
Relate your observation to the idea of statistical precision.

---

## Exercise 6
Modify the parameters of the HV curve:
- `hv50`
- `k`
- `plateau`

Explain what each parameter changes in the shape of the curve.

---

## Exercise 7
Add a second coordinate `y` and define a 2D matching criterion using both `x` and `y`.

---

## Exercise 8
Write a short paragraph, in your own words, answering this question:

**What is the conceptual meaning of an eligible event, and why must it be defined carefully when computing detector efficiency?**

# Final remark

This notebook is a bridge between detector physics and scientific programming.  
Its main objective is not only to compute a number, but to help students understand how a detector-performance observable is constructed from geometric and timing logic.

A good next step would be:
- to turn one of the exercises into a mini-project,
- or to connect this toy model to real experimental data in a more advanced course.